In [1]:
pip install transformers datasets seqeval pandas torch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e34c654906e7eb2f82b29d40a1f23d9d4e87981ed7faec511d45a9e0b9069db0
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [6]:
import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
from seqeval.metrics import f1_score


# Load dataset

df = pd.read_csv("Full_NER_Dataset.csv")
df = df.dropna(subset=["Token"])

tokens = df["Token"].astype(str).tolist()
tags = df["Entity tag"].astype(str).tolist()
CHUNK_SIZE = 32

sentences = [
    tokens[i:i+CHUNK_SIZE]
    for i in range(0, len(tokens), CHUNK_SIZE)
]

labels = [
    tags[i:i+CHUNK_SIZE]
    for i in range(0, len(tags), CHUNK_SIZE)
]


# Label mapping

label_list = sorted(set(tags))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

dataset = Dataset.from_dict({
    "tokens": sentences,
    "ner_tags": [[label2id[t] for t in seq] for seq in labels]
})

dataset = dataset.train_test_split(test_size=0.2, seed=42)


# Tokenization

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_and_align(batch):
    tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    aligned_labels = []
    for i, label_seq in enumerate(batch["ner_tags"]):
        word_ids = tokenized.word_ids(i)
        prev = None
        label_ids = []

        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                label_ids.append(label_seq[w])
            else:
                label_ids.append(-100)
            prev = w

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

dataset = dataset.map(tokenize_and_align, batched=True)


# Model

model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer)


# Metrics

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true, pred = [], []
    for p_seq, l_seq in zip(preds, labels):
        t, pr = [], []
        for pi, li in zip(p_seq, l_seq):
            if li != -100:
                t.append(id2label[li])
                pr.append(id2label[pi])
        true.append(t)
        pred.append(pr)

    return {"f1": f1_score(true, pred)}


# Training

training_args = TrainingArguments(
    output_dir="./xlmr_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    fp16=True,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())


Map:   0%|          | 0/2847 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3413776956.py:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
100,0.845600
200,0.259800
300,0.171900
400,0.150500
500,0.125100


{'eval_loss': 0.1371772587299347, 'eval_f1': 0.7795135455803835, 'eval_runtime': 1.1638, 'eval_samples_per_second': 611.782, 'eval_steps_per_second': 38.666, 'epoch': 3.0}


In [7]:
from seqeval.metrics import classification_report
import numpy as np

predictions, labels, _ = trainer.predict(dataset["test"])

preds = np.argmax(predictions, axis=2)

true_labels = []
pred_labels = []

for p_seq, l_seq in zip(preds, labels):
    t_seq = []
    p_seq_out = []
    for p, l in zip(p_seq, l_seq):
        if l != -100:
            t_seq.append(id2label[l])
            p_seq_out.append(id2label[p])
    true_labels.append(t_seq)
    pred_labels.append(p_seq_out)

print(classification_report(true_labels, pred_labels))


              precision    recall  f1-score   support

   BODY_PART       0.85      0.82      0.83       532
     DISEASE       0.83      0.76      0.79       371
  MEDICATION       0.72      0.66      0.69       408
   PROCEDURE       0.82      0.79      0.81       417
     SYMPTOM       0.75      0.79      0.77       821

   micro avg       0.79      0.77      0.78      2549
   macro avg       0.79      0.77      0.78      2549
weighted avg       0.79      0.77      0.78      2549

